In [1]:
import pandas as pd

In [2]:
# 1. KIỂM TRA CARD ĐỒ HỌA (GPU)
print("========== 1. THÔNG TIN GPU ==========")
!nvidia-smi

import torch
print("\n========== 2. TRẠNG THÁI PYTORCH ==========")
print(f"PyTorch đã nhận GPU chưa?: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Đang sử dụng Card: {torch.cuda.get_device_name(0)}")
    print(f"Dung lượng VRAM: {round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)} GB")

# 3. KIỂM TRA CPU VÀ RAM HỆ THỐNG
print("\n========== 3. THÔNG TIN CPU & RAM ==========")
print("CPU:")
!lscpu | grep 'Model name'
!lscpu | grep '^CPU(s):'
print("\nRAM Hệ thống:")
!free -h

========== 1. THÔNG TIN GPU ==========
Fri Mar 13 09:49:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!mkdir -p /content/drive/MyDrive/HALO_Train_Data

In [7]:
# 1. Kết nối Colab với Drive của bạn (nó sẽ hiện bảng hỏi quyền, bấm Cho phép)
from google.colab import drive
drive.mount('/content/drive')

MessageError: Failed to issue request POST https://colab.research.google.com/tun/m/credentials-propagation/gpu-t4-s-kkb-ass1c2-1d8ga5y8fapfh?authtype=dfs_ephemeral&version=2&dryrun=false&propagate=true&record=false&authuser=0&authuser=0: Bad Request
Response body: 
<!DOCTYPE html>
<html lang=en>
  <meta charset=utf-8>
  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">
  <title>Error 400 (Bad Request)!!1</title>
  <style>
    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/logos/errorpa

In [9]:
# Đã thêm HALO_Train_Data vào đường dẫn
!cp /content/drive/MyDrive/HALO_Train_Data/amr_ws.zip /content/
!unzip -q /content/amr_ws.zip -d /content/

In [6]:
%%bash
set -euo pipefail

LOG_FILE=/content/halo_setup_train.log
rm -f "${LOG_FILE}"
exec > >(tee -a "${LOG_FILE}") 2>&1

HALO_DIR=/content/amr_ws/HALO
if [ ! -d "${HALO_DIR}" ]; then
  echo "ERROR: Không thấy ${HALO_DIR}. Hãy chạy cell giải nén amr_ws.zip trước."
  exit 1
fi

echo "========== APT =========="
apt-get update -y
apt-get install -y build-essential cmake git libopencv-dev libeigen3-dev python3-dev pkg-config

echo "========== PIP =========="
python3 -m pip install -U pip setuptools wheel
python3 -m pip install "numpy<2" gym==0.26.2 tensorboard tensorboardx pyglet==1.5.15 socialforce casadi pybind11
python3 -m pip install --index-url https://download.pytorch.org/whl/cu121 torch torchvision
python3 -m pip install dgl -f https://data.dgl.ai/wheels/cu121/repo.html || python3 -m pip install dgl

echo "========== BUILD OCP =========="
cd "${HALO_DIR}/src/ocp_planner"
mkdir -p extern
if [ ! -d extern/pybind11 ]; then
  git clone https://github.com/pybind/pybind11.git extern/pybind11
fi
cp CMakeLists_standalone.txt CMakeLists.txt

PY_EXE=$(which python3)
PY_INC=$(${PY_EXE} - <<'PY'
import sysconfig
print(sysconfig.get_paths()['include'])
PY
)
CASADI_DIR=$(${PY_EXE} - <<'PY'
import casadi, pathlib
print(pathlib.Path(casadi.__file__).resolve().parent)
PY
)
CASADI_LIB=$(${PY_EXE} - <<'PY'
import casadi, pathlib, glob
d = pathlib.Path(casadi.__file__).resolve().parent
cands = sorted(glob.glob(str(d / 'libcasadi.so*')))
print(cands[-1] if cands else '')
PY
)

if [ -z "${CASADI_LIB}" ]; then
  echo "ERROR: libcasadi.so not found in pip casadi package"
  exit 1
fi

sed -i "s|set(CASADI_INCLUDE_DIR.*|set(CASADI_INCLUDE_DIR \"${CASADI_DIR}/include\")|g" CMakeLists.txt
sed -i "s|set(CASADI_LIB_DIR.*|set(CASADI_LIB_DIR \"${CASADI_DIR}\")|g" CMakeLists.txt
sed -i "s|set(PYTHON_EXECUTABLE.*|set(PYTHON_EXECUTABLE \"${PY_EXE}\")|g" CMakeLists.txt
sed -i "s|set(PYTHON_INCLUDE_DIRECTORY.*|set(PYTHON_INCLUDE_DIRECTORY \"${PY_INC}\")|g" CMakeLists.txt

build_once () {
  local EXTRA_CXX_FLAGS="$1"
  rm -rf build
  mkdir -p build
  cd build
  cmake .. -DCMAKE_BUILD_TYPE=Release -DCMAKE_CXX_FLAGS="${EXTRA_CXX_FLAGS}"
  make -j"$(nproc)"
  make install
  cd ..
}

export LD_LIBRARY_PATH="${CASADI_DIR}:${LD_LIBRARY_PATH:-}"
build_once ""

cd "${HALO_DIR}/drl_moudle"
if ! python3 -c "import ocp_planner_py; print('ocp_planner_py OK (default ABI)')"; then
  echo "Import failed -> retry ABI=0"
  cd "${HALO_DIR}/src/ocp_planner"
  build_once "-D_GLIBCXX_USE_CXX11_ABI=0"
  cd "${HALO_DIR}/drl_moudle"
  python3 -c "import ocp_planner_py; print('ocp_planner_py OK (ABI=0)')"
fi

echo "[OK] Setup + build thành công. Log: ${LOG_FILE}"

ERROR: Không thấy /content/amr_ws/HALO. Hãy chạy cell giải nén amr_ws.zip trước.


CalledProcessError: Command 'b'set -euo pipefail\n\nLOG_FILE=/content/halo_setup_train.log\nrm -f "${LOG_FILE}"\nexec > >(tee -a "${LOG_FILE}") 2>&1\n\nHALO_DIR=/content/amr_ws/HALO\nif [ ! -d "${HALO_DIR}" ]; then\n  echo "ERROR: Kh\xc3\xb4ng th\xe1\xba\xa5y ${HALO_DIR}. H\xc3\xa3y ch\xe1\xba\xa1y cell gi\xe1\xba\xa3i n\xc3\xa9n amr_ws.zip tr\xc6\xb0\xe1\xbb\x9bc."\n  exit 1\nfi\n\necho "========== APT =========="\napt-get update -y\napt-get install -y build-essential cmake git libopencv-dev libeigen3-dev python3-dev pkg-config\n\necho "========== PIP =========="\npython3 -m pip install -U pip setuptools wheel\npython3 -m pip install "numpy<2" gym==0.26.2 tensorboard tensorboardx pyglet==1.5.15 socialforce casadi pybind11\npython3 -m pip install --index-url https://download.pytorch.org/whl/cu121 torch torchvision\npython3 -m pip install dgl -f https://data.dgl.ai/wheels/cu121/repo.html || python3 -m pip install dgl\n\necho "========== BUILD OCP =========="\ncd "${HALO_DIR}/src/ocp_pla

In [3]:
!nvidia-smi


Fri Mar 13 16:42:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [23]:
# Cell 8 - Chẩn đoán nhanh lỗi setup/train gần nhất
import os
log_file = '/content/halo_setup_train.log'
print('Log exists:', os.path.exists(log_file))
if os.path.exists(log_file):
    print('\n===== STEP MARKERS =====')
    !grep -n "^==========" /content/halo_setup_train.log || true
    print('\n===== ABI CHECK =====')
    !grep -nE "default ABI|ABI=0|Import failed|ocp_planner_py OK" /content/halo_setup_train.log || true
    print('\n===== LAST 100 LINES =====')
    !tail -n 100 /content/halo_setup_train.log
    print('\n===== KEY ERROR LINES =====')
    !grep -nEi "error|failed|not found|undefined symbol|CMake Error|Traceback" /content/halo_setup_train.log | tail -n 60 || true
else:
    print('Chưa có log. Hãy chạy Cell 7 trước.')

Log exists: True

===== STEP MARKERS =====

===== LAST 80 LINES =====
Using PY_EXE=/usr/bin/python3
Using PY_INC=/usr/include/python3.12
Using CASADI_DIR=/usr/local/lib/python3.12/dist-packages/casadi
\n========== CMake configure ==========
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found OpenCV: /usr (found version "4.5.4")
-- Found Casadi: /usr/local/lib/python3.12/dist-packages/casadi/libcasadi.so
-- pybind11 v3.1.0 
-- Found Python: /usr/local/bin/python (found suitable version "3.12.12", minimum required is "3.8") found comp

## Train HALO PPO trên Colab
Chạy cell bên dưới để train chính thức. Có thể đổi `OUTPUT_DIR`, `TOTAL_TIMESTEPS`, `EVAL_FREQ`, `N_EVAL_EPISODES` theo nhu cầu.

In [ ]:
%%bash
set -euo pipefail

cd /content/amr_ws/HALO/drl_moudle
export LD_LIBRARY_PATH="$(python3 - <<'PY'
import casadi, pathlib
print(pathlib.Path(casadi.__file__).resolve().parent)
PY
):${LD_LIBRARY_PATH:-}"

python3 train_ppo.py \
  --config configs/mpc_rl.py \
  --output_dir train_data/colab_run \
  --total_timesteps 5000000 \
  --eval_freq 500 \
  --n_eval_episodes 100 \
  --action_dim 9 \
  --action_range 2.25 \
  --use_AM True \
  --use_PL True \
  --n_steps 4096 \
  --batch_size 256 \
  --n_epochs 10

In [ ]:
%%bash
set -euo pipefail

cd /content/amr_ws/HALO/drl_moudle
export LD_LIBRARY_PATH="$(python3 - <<'PY'
import casadi, pathlib
print(pathlib.Path(casadi.__file__).resolve().parent)
PY
):${LD_LIBRARY_PATH:-}"

python3 train_ppo.py \
  --config configs/mpc_rl.py \
  --output_dir train_data/colab_run \
  --resume \
  --total_timesteps 5000000 \
  --eval_freq 500 \
  --n_eval_episodes 100 \
  --action_dim 9 \
  --action_range 2.25 \
  --use_AM True \
  --use_PL True \
  --n_steps 4096 \
  --batch_size 256 \
  --n_epochs 10

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/amr_ws/HALO/drl_moudle/train_data/colab_run/PPO_1 --port 6006